# Category Classifiers — Model Development and Corpus Scoring

> **Sanitized release.** This notebook is a faithful copy of the original working
> notebook from a federal document-classification project, edited for public
> sharing: cell outputs are stripped, file paths and system names are
> generalized, and the ten real business categories are renamed `CAT-A` … `CAT-J`.
> Cells that were duplicated once per category in the original are collapsed to
> one or two representative copies, marked with a note. Nothing else about the
> structure or approach has been changed.


The notebook below is a Jupyter Notebook. Jupyter Notebook is a web-based
interactive computing platform. The notebook combines live code, equations,
narrative text, visualizations, and text.

## Table of Contents:

- [1. Research Question](#1.)
- [2.1 Pre-process Data](#2.1)
    - [2.2 Load dataset, generate record_flag and file_name](#2.2)
    - [2.3 Extract document types from filename field](#2.3)
    - [2.4 Remove extra white space from documents](#2.4)
    - [2.5 Build document flag](#2.5)
- [3.1 Train the model](#3.1)
- [3.2 Deploy the trained model on record-number files](#3.2)

<a id='1.'></a>
## 1. Research Question

Can a classification model be built to identify individual business categories of documents in the repository?

<a id='2.1'></a>
## 2.1 Pre-Process Data

In this section the page-extraction data is loaded and prepared for downstream
use in the classification model.

The extraction contains 4 fields: document_id, directory, old_ocr and new_ocr:

- document_id: my personal primary key, used to quickly index and transfer documents across various project stages.
- directory: the full path, including filename
- old_ocr: the string output of the original ocr of the file (the legacy text layer embedded whenever the document was first digitized), first pages.
- new_ocr: the string output of my deployment of Tesseract OCR on the same file. Also first pages.

<a id='2.2'></a>
## 2.2 Load dataset, generate record_flag and file_name.

The section below generates a record_flag to partition the dataset into two categories: 1) data used to train models and 2) data to deploy trained models on.

record_flag data dictionary:
- 1: an opaque record-number filename, set aside for later scoring.
- 0: a filename carrying type-code metadata, used in training the model.

In [ ]:
# generate record_flag, file_name

#import modules
import re
import pandas as pd
from tqdm import tqdm

#load dataset
df = pd.read_csv('master_page_extraction.csv')

#generate lists
document_id = df['document_id'].tolist()
pdf_dir = df['directory'].tolist()
old_ocr = df['old_ocr'].tolist()
new_ocr = df['new_ocr'].tolist()

#the repository's record-number filename convention, generalized for release
RECORD_ID_PATTERN = re.compile(r'^[A-Z]\d{5,}')

#find record-number file names and assign flag
record_flag = []
file_name = []
for d in tqdm(pdf_dir):
    name = d.split('/')[-1]  #select filename
    file_name.append(name)
    record_flag.append(1 if RECORD_ID_PATTERN.match(name) else 0)

# add record_flag, file_name to df
df = pd.DataFrame({'document_id': document_id,
                   'record_flag': record_flag,
                   'directory': pdf_dir,
                   'file_name': file_name,
                   'old_ocr': old_ocr,
                   'new_ocr': new_ocr})

#save to csv
df.to_csv('model_recordflag.csv', index=False)

print("|INFO| Generated record_flag, file_name columns.")

<a id='2.3'></a>
## 2.3 Extract document types from filename field

Decades of humans encoded document types into filenames — inconsistently, with
typos and drift. This section mines those conventions into a doc_type column.
The extraction was built with the intention of being exhaustive across all
document types.

*Collapsed: the original cell checked ~40 real type codes and their observed
misspellings in one long if-chain, followed by dozens of hand-written
edge-case cleanup rules. The structure below is identical; the code list is
representative.*

In [ ]:
# extract document types from filename field

#load file from previous step
df = pd.read_csv('model_recordflag.csv')
file_name = df['file_name'].tolist()

#known type codes and the misspelled variants observed in real filenames
TYPE_CODE_VARIANTS = {
    'CAT-A': ['CAT-A', 'CATA', 'CAT_A', 'CAT A'],
    'CAT-B': ['CAT-B', 'CATB', 'CTA-B'],
    'CAT-C': ['CAT-C', 'CATC'],
    # ...continues for every category and auxiliary code (~40 total)
}

doc_type = []
for f in tqdm(file_name):
    f = f.upper()
    found_doc_types = []
    for canonical, variants in TYPE_CODE_VARIANTS.items():
        if any(v in f for v in variants):
            found_doc_types.append(canonical)
    if RECORD_ID_PATTERN.match(f):
        found_doc_types.append('RECORD')
    if not found_doc_types:
        found_doc_types.append('NOT CLASSIFIED')

    # sort results, remove duplicates, and then join
    found_doc_types = sorted(set(found_doc_types))
    found_doc_types = '-'.join(found_doc_types)

    #edge-case cleanup (representative examples of ~30 hand rules)
    if found_doc_types == 'CAT-A-RECORD':
        found_doc_types = 'RECORD'
    if found_doc_types == 'CAT-A-CAT-A':
        found_doc_types = 'CAT-A'

    doc_type.append(found_doc_types)

df['doc_type'] = doc_type
df.to_csv('model_recordflag_doctype.csv', index=False)

print("|INFO| Generated doc_type column.")

<a id='2.4'></a>
## 2.4 Remove extra white space from documents

In [ ]:
#remove extra white space from documents, my ocd
#load csv from previous step
df = pd.read_csv('model_recordflag_doctype.csv')
new_ocr = df['new_ocr'].tolist()
old_ocr = df['old_ocr'].tolist()

# remove >= 2 spaces between "words"
temp_new_ocr = []
for doc in tqdm(new_ocr):
    doc = " ".join(doc.split())
    temp_new_ocr.append(doc)

temp_old_ocr = []
for doc in tqdm(old_ocr):
    doc = " ".join(doc.split())
    temp_old_ocr.append(doc)

df['new_ocr'] = temp_new_ocr
df['old_ocr'] = temp_old_ocr

df.to_csv('model_recordflag_doctype_onespace.csv', index=False)

print("|INFO| White space removed.")

<a id='2.5'></a>
## 2.5 Build doc flag.

The code below builds a binary flag per category, used to train that
category's model. Condensing related type codes into a single category was
presented to program staff with the benefits/drawbacks rationale before
adoption.

Each category got **two labeling variants**, trained and compared:

- **general** — the type code appears anywhere in the filename's doc_type
- **specific** — the doc_type is exactly that code

Data dictionary for the doc flag:
- 1: filename contained the doc words
- 0: filename did not contain the doc words

*Collapsed: one general + one specific builder is shown for CAT-A; the
original repeated this pair for every category.*

In [ ]:
#build binary classifier flag for CAT-A (general)
df = pd.read_csv('model_recordflag_doctype_onespace.csv')
doc_type = df['doc_type'].tolist()
doc_type_numeric = []
for doc in tqdm(doc_type):
    if 'CAT-A' in doc:
        doc_type_numeric.append(1)
    else:
        doc_type_numeric.append(0)

df['doc_type_numeric'] = doc_type_numeric
#save new csv
df.to_csv('model_recordflag_doctype_onespace_CAT-A_general.csv', index=False)

print("|INFO| Doc flag built.")

In [ ]:
#build binary classifier flag for CAT-A (specific)
df = pd.read_csv('model_recordflag_doctype_onespace.csv')
doc_type = df['doc_type'].tolist()
doc_type_numeric = []
for doc in tqdm(doc_type):
    if doc == 'CAT-A':
        doc_type_numeric.append(1)
    else:
        doc_type_numeric.append(0)

df['doc_type_numeric'] = doc_type_numeric
#save new csv
df.to_csv('model_recordflag_doctype_onespace_CAT-A_specific.csv', index=False)

print("|INFO| Doc flag built.")

### Metadata-database cross-reference (CAT-A only)

For one category, filename mining could be augmented: the review team
maintained a small metadata database whose entries identified confirmed
CAT-A documents. The original read a Microsoft Access backend over pyodbc —
backing it up before every read — and merged those confirmed paths into the
CAT-A labels. (Paths, database name, and field names generalized.)

In [ ]:
#build binary classifier for CAT-A PART 1
#add the metadata database's manual classifications to the flag (ONLY for the CAT-A classifier)
from shutil import copyfile
import pyodbc
import time
import pandas as pd

def access_backup():
    #save copy of access db as backup
    print('|INFO| backing up access db.')
    current_day = time.strftime("%m%d%Y")
    filepath = 'metadata_db/db/review-metadata-be.accdb'
    backup_path = '{}{}{}'.format('metadata_db/backups/automated/review-metadata-be-', current_day, '.accdb')
    copyfile(filepath, backup_path)
    print('|INFO| access db backup complete.')
    return backup_path

def extract_table(backup_path):
    print('|INFO| extracting df from access database.')
    myDataSources = pyodbc.dataSources()
    access_driver = myDataSources['MS Access Database']
    cnxn = pyodbc.connect(driver=access_driver, dbq=backup_path, autocommit=True)

    table_name = 'extracted'
    query = 'SELECT DISTINCT category_a_status, file_path FROM {}'.format(table_name)
    df = pd.read_sql(query, cnxn)
    print('|INFO| extracted df from backup access database.')
    return df

#run above functions
backup_path = access_backup()
df_meta = extract_table(backup_path)

#subset by confirmed-document statuses
df_meta_confirmed = df_meta[df_meta['category_a_status'].isin(['Addendum', 'Draft', 'Original', 'Updated'])]

#make list of confirmed paths and save
confirmed_pdf_dir = df_meta_confirmed['file_path'].tolist()
print('confirmed CAT-A documents:')
print(len(confirmed_pdf_dir))

confirmed_pdf_dir = pd.DataFrame(list(confirmed_pdf_dir), columns=['confirmed_pdf_dir'])
confirmed_pdf_dir.to_csv('confirmed_pdf_dir.csv', index=False)

In [ ]:
#PART 2: merge the confirmed CAT-A paths into the flag
from tqdm import tqdm
import pandas as pd
df = pd.read_csv('model_recordflag_doctype_onespace.csv')
confirmed_pdf_dir = pd.read_csv('confirmed_pdf_dir.csv')
confirmed_pdf_dir = confirmed_pdf_dir['confirmed_pdf_dir'].tolist()

pdf_dir = df['directory'].tolist()
doc_type = df['doc_type'].tolist()
print(doc_type.count('CAT-A'))
new_doc_type_numeric = []
temp_doc_type = []
for pdf, doc in tqdm(zip(pdf_dir, doc_type)):
    if pdf in confirmed_pdf_dir:      #confirmed by the metadata database
        temp_doc_type.append('CAT-A')
        new_doc_type_numeric.append(1)
    elif 'CAT-A' in doc:              #metadata extracted from filename
        temp_doc_type.append(doc)
        new_doc_type_numeric.append(1)
    else:
        temp_doc_type.append(doc)
        new_doc_type_numeric.append(0)

print(new_doc_type_numeric.count(1))

df['doc_type_numeric'] = new_doc_type_numeric
df['doc_type'] = temp_doc_type

#save new csv
df.to_csv('model_recordflag_doctype_onespace_CAT-A_general.csv', index=False)

<a id='3.1'></a>
## 3.1 Train the actual model.

(and)

<a id='3.2'></a>
## 3.2 Deploy the trained model on the ENTIRE repository!

One cell per category run: set `var`, pick that category's tuned model from
the block below, train on the labeled (non-record-number) documents, then
score every document in the corpus with `predict` and `predict_proba`.

The commented block preserves the real per-category tuning log — regularization
strength found by randomized search, and held-out accuracy across labeling
variants and oversampling strategies (`os=`). Only the category names are
changed.

In [ ]:
var = 'CAT-A'
model_subtype = 'specific'

#log model
from sklearn.model_selection import train_test_split #for split the data
from sklearn.metrics import accuracy_score  #for accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import RandomOverSampler
import pandas as pd
from tqdm import tqdm
import numpy as np

#load file from previous step
df = pd.read_csv('model_recordflag_doctype_onespace_{}_{}.csv'.format(var, model_subtype))
new_ocr = df['new_ocr'].tolist()
old_ocr = df['old_ocr'].tolist()

#join old_ocr and new_ocr into giant string
#(weak text in one stream can be rescued by the other)
temp_lst = []
for new, old in zip(new_ocr, old_ocr):
    temp_doc = new + ' ' + old
    temp_lst.append(temp_doc)
df['super_ocr'] = temp_lst
temp_lst = None
super_ocr = df['super_ocr'].tolist()

#split record-number docs (deploy set) from labeled docs (training pool)
df_record = df[df['doc_type'] == 'RECORD']
df_labeled = df[df['doc_type'] != 'RECORD']
df_record = None
df = None
old_ocr = None
new_ocr = None

#separate prediction value and the rest of the dataset
X = df_labeled.drop('doc_type_numeric', 1)
y = df_labeled['doc_type_numeric']
df_labeled = None #remove old training set from memory

#oversample parameters
oversample = RandomOverSampler(sampling_strategy='minority', random_state=42)

#execute oversampling
X_over, y_over = oversample.fit_resample(X, y)
X = None
y = None

#add generated doc_type_numeric to the main df
X_over['doc_type_numeric'] = y_over
df_labeled = X_over
X_over = None
y_over = None

#test-train-split, vectorize and fit.
X_train, X_test, y_train, y_test = train_test_split(
    df_labeled.super_ocr, df_labeled.doc_type_numeric, test_size=0.1, random_state=42)
df_labeled = None
vect = CountVectorizer(ngram_range=(1, 3), stop_words='english')
X_train = vect.fit_transform(X_train)
X_test = vect.transform(X_test)

#CAT-A
model = LogisticRegression(C=375.4694224073337, max_iter=1000, multi_class='ovr',
                   penalty='l1', solver='liblinear')
                   # ngram (2,2) 98.79
                   # V4 ngram (1,3) 98.62 specific
                   # V4 ngram (1,3) 98.66 general
                   # V4.1 os=0.5 99.17 specific
                   # V4.1 os=0.5 99.21 general
                   # V4.1 os='minority' 99.18 specific
                   # V4.1 os='minority' 99.39 general

#CAT-B
#model = LogisticRegression(C=3072.4688427090036, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.36 specific
                   # V4 ngram (1,3) 99.38 general
                   # V4.1 os=0.5 99.49 specific
                   # V4.1 os=0.5 99.56 general

#CAT-C
#model = LogisticRegression(C=845.1366330684722, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.47 general
                   # V4 ngram (1,3) 99.41 specific
                   # V4.1 os='minority' 99.49 specific
                   # V4.1 os='minority' 99.50 general

#CAT-D
#model = LogisticRegression(C=13.095350204826676, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.71 specific
                   # V4 ngram (1,3) 99.68 general
                   # V4.1 os='minority' 99.40 specific
                   # V4.1 os='minority' 99.51 general

#CAT-E
#model = LogisticRegression(C=15.176833902834039, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.81 specific
                   # V4 ngram (1,3) 99.79 general
                   # V4.1 os=0.25 99.85 specific
                   # V4.1 os=0.25 99.76 general

#CAT-F
#model = LogisticRegression(C=55.17492376129129, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.33 general
                   # V4 ngram (1,3) 99.44 specific
                   # V4.1 os=0.5 99.19 specific
                   # V4.1 os=0.5 99.19 general

#CAT-G
#model = LogisticRegression(C=5.019965133110079, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.39 specific
                   # V4 ngram (1,3) 99.51 general
                   # V4.1 os='minority' 99.45 specific
                   # V4.1 os='minority' 99.43 general

#CAT-H
#model = LogisticRegression(C=279.5415999067859, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # ngram (1,3) 99.68 specific
                   # ngram (1,3) 99.68 general
                   # V4.1 os='minority' 99.38 specific
                   # V4.1 os='minority' 99.27 general

#CAT-I
#model = LogisticRegression(C=4962.444877628914, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # ngram (1,3) 99.9 specific
                   # ngram (1,3) 99.2 general
                   # V4.1 os='minority' 99.51 specific
                   # V4.1 os='minority' 99.49 general

#CAT-J
#model = LogisticRegression(C=4.0983836717572615, max_iter=1000, multi_class='ovr',
#                   penalty='l1', solver='liblinear')
                   # V4 ngram (1,3) 99.78 specific
                   # V4 ngram (1,3) 99.60 general
                   # V4.1 os='minority' 99.39 specific
                   # V4.1 os='minority' 99.38 general

model.fit(X_train, y_train)
prediction_lr = model.predict(X_test)
score = round(accuracy_score(prediction_lr, y_test) * 100, 2)
print(score)
#PART 2------------------------------------------------------------------------------------
#deploy model on entire repository:

#predict on all
temp_prob = []
temp_pred = []
for doc in tqdm(super_ocr):
    test = vect.transform([doc])

    #pred
    pred = model.predict(test)
    temp_pred.append(str(pred[0]))
    #prob
    prob = model.predict_proba(test)
    temp_prob.append(str(float(np.round(prob[0][1], 9))))

#build spreadsheet hyperlink column so reviewers can click straight to a document
df_all = pd.read_csv('model_recordflag_doctype_onespace_{}_{}.csv'.format(var, model_subtype))
document_id = df_all['document_id'].tolist()
pdf_dir = df_all['directory'].tolist()
record_flag = df_all['record_flag'].tolist()
doc_type = df_all['doc_type'].tolist()
hyperlink = []
for dir in pdf_dir:
    lnk = '{}{}{}'.format('=HYPERLINK(SUBSTITUTE("', dir, '", "/", "\\"),"link")')
    hyperlink.append(lnk)

#move results to pd df
df_pred = pd.DataFrame(
    list(zip(document_id, pdf_dir, hyperlink, record_flag, doc_type, temp_pred, temp_prob)),
    columns=['document_id', 'directory', 'hyperlink', 'record_flag', 'doc_type', 'pred', 'prob'])

#save as csv
df_pred.to_csv('pred_LR_{}_{}.csv'.format(var, model_subtype), index=False)

print("|INFO| {} {} Predictions Generated.".format(var, model_subtype))

Held-out evaluation for the category just trained:

In [ ]:
from sklearn.metrics import classification_report
target_names = ['not_{}'.format(var), '{}'.format(var)]
print(classification_report(y_test, prediction_lr, target_names=target_names))
# representative category result (from the original run):
#               precision    recall  f1-score   support
#  not_CAT-A         0.99      0.99      0.99      5568
#      CAT-A         0.99      0.99      0.99      5303
#   accuracy                             0.99     10871

In [ ]:
#ref: https://neuralnetlab.com/auc-sklearn-with-practical-example/
#training figures
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt
# Calculating the prediction values and assigning them into a variable
pred_val = model.predict_proba(X_test)

# roc curve for the logistic regression model
fpr, tpr, threshold = roc_curve(y_test, pred_val[:, 1], pos_label=1)

# Calculating the curve where tpr = fpr
random_pred_val = [0 for i in range(len(y_test))]
p_fpr1, p_tpr1, _ = roc_curve(y_test, random_pred_val, pos_label=1)

# Calculating the auc score  (representative category: 0.9992)
auc_score = roc_auc_score(y_test, pred_val[:, 1])
print(auc_score)

#the figure
plt.style.use('seaborn')
plt.plot(fpr, tpr, linestyle='--', color='red', label='Logistic Regression: {}_{}'.format(var, model_subtype))
plt.plot(p_fpr1, p_tpr1, linestyle='--', color='black')
plt.title('ROC curve plot: {}_{}'.format(var, model_subtype))
plt.xlabel('False Positive Rate/FPR')
plt.ylabel('True Positive Rate/TPR')
plt.legend(loc='best')
plt.savefig('ROC curve plot {}_{}'.format(var, model_subtype), dpi=300)

### Hyper-parameter optimization code (used previously)

In [ ]:
#hyper-parameter optimizer
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
model = LogisticRegression()
param_grid = [
    {
    'penalty': ['l1'],
    'solver': ['liblinear'],
    'multi_class': ['ovr'],
    'max_iter': [1000],
    'C': np.logspace(-4, 4, 1000)
    }
]

clf = RandomizedSearchCV(model, param_distributions=param_grid, cv=2, n_iter=10, verbose=10, n_jobs=1)

best_clf = clf.fit(X_train, y_train)

#print params found from optimizer
print(best_clf.best_params_)

#print scores from optimizer
print(best_clf.best_score_)

In [ ]:
#alt logreg optimizer
#does not work
#https://github.com/huawei-noah/HEBO/tree/master/HEBO
from sklearn.metrics import r2_score
from hebo.sklearn_tuner import sklearn_tuner

space_cfg = [
    {'name': 'penalty',     'type': 'cat', 'categories': ['l1']},
    {'name': 'C',           'type': 'int', 'lb': 1, 'ub': 1000},
    {'name': 'solver',      'type': 'cat', 'categories': ['liblinear']},
    {'name': 'multi_class', 'type': 'cat', 'categories': ['ovr']},
    {'name': 'max_iter',    'type': 'int', 'lb': 1000, 'ub': 1000}
]

result = sklearn_tuner(LogisticRegression, space_cfg, X_train, y_train, metric=r2_score, max_iter=16)

In [ ]:
#save model
from joblib import dump, load
dump(model, 'cat-a-specific_ngram_one_three.joblib')

#load model
#model = load('cat-a-specific_ngram_one_three.joblib')

# Prediction Cleanup

Overwrite prediction results for documents whose filenames already carried the
type code — those are *known from metadata*, not model discoveries, so their
probability field gets a `KNOWN` sentinel. Reviewers can then always tell
discovery apart from confirmation.

*Collapsed: shown for two categories; the original repeated the block for all.*

In [ ]:
#open the merged all-category prediction master
import pandas as pd
df = pd.read_csv('pred_LR_master_general_cleaned_input.csv')
doc_type = df['doc_type'].tolist()

for cat in ['CAT-A', 'CAT-B']:  # ...ran across every category in the original
    pred_col = df['pred_{}'.format(cat)].tolist()
    prob_col = df['prob_{}'.format(cat)].tolist()
    temp_pred = []
    temp_prob = []
    for d, pred, prob in tqdm(zip(doc_type, pred_col, prob_col)):
        if cat in d:
            temp_pred.append(1)
            temp_prob.append('KNOWN')
        else:
            temp_pred.append(pred)
            temp_prob.append(prob)
    df['pred_{}'.format(cat)] = temp_pred
    df['prob_{}'.format(cat)] = temp_prob

df.to_csv('pred_LR_master_general_cleaned.csv', index=False)

# No Pred

If a document produced no `pred == 1` from any model, return the max
probability across models anyway — so even the "no prediction" pile arrives
ranked for review instead of alphabetized.

In [ ]:
import pandas as pd
from tqdm import tqdm

df = pd.read_csv('pred_LR_master_general_cleaned_nopred.csv')

prob_cols = ['prob_CAT-A', 'prob_CAT-B', 'prob_CAT-C', 'prob_CAT-D', 'prob_CAT-E',
             'prob_CAT-F', 'prob_CAT-G', 'prob_CAT-H', 'prob_CAT-I', 'prob_CAT-J']

#return top prob value from zero_pred file
top_prob = []
top_type = []
for row in tqdm(df[prob_cols].itertuples(index=False)):
    vals = list(row)
    idx = vals.index(max(vals))
    top_type.append(prob_cols[idx])
    top_prob.append(max(vals))

df['top_prob'] = top_prob
df['top_type'] = top_type

df.to_csv('pred_LR_master_general_no_pred_ranked.csv', index=False)

df.top_type.value_counts()

# Plots

Inspect distribution of probabilities in each logreg model, over the unlabeled
record-number documents — general vs specific labeling variants, log scale.
The result was strongly bimodal: decisive mass near 0 and 1, few ambiguous
middles, which is what makes probability ranking work as a triage tool.
Sanitized recreation: [`../figures/probability-distribution.svg`](../figures/probability-distribution.svg).

*Collapsed: one probability-distribution cell and one prediction-count cell are
shown; the original repeated the pair per category.*

In [ ]:
#CAT-A
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
%matplotlib inline
plt.style.use('seaborn-deep')

var = 'prob_CAT-A'

#load and preprocess data for gen figure
df = pd.read_csv('pred_LR_master_general_cleaned.csv', low_memory=False)
df_g = df[df['doc_type'] == 'RECORD'] #subset
df_g = df_g.reset_index(drop=True)
df_g = df_g[var].astype('float64')
df = None

#load and preprocess data for spec figure
df = pd.read_csv('pred_LR_master_specific_cleaned.csv', low_memory=False)
df_s = df[df['doc_type'] == 'RECORD'] #subset
df_s = df_s.reset_index(drop=True)
df_s = df_s[var].astype('float64')
df = None

# Plot Histogram on x
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 100})
plt.hist([df_g, df_s], bins=20, label=['General', 'Specific'])
plt.gca().set(title='Probability Distribution: {}'.format(var), ylabel='Frequency')
plt.legend(loc='upper right', title='Model Class:', fontsize='small')
plt.xticks(np.arange(0, 1.1, step=0.1))
plt.xlabel('Probability Score (0-1)')
plt.yscale('log')

In [ ]:
#CAT-A
var = 'pred_CAT-A'
#load data for figure:
#gen
df = pd.read_csv('pred_LR_master_general_cleaned.csv', low_memory=False)
df_g = df[df['doc_type'] == 'RECORD']
df_g = df_g[df_g[var] == 1]
df = None
#spe
df = pd.read_csv('pred_LR_master_specific_cleaned.csv', low_memory=False)
df_s = df[df['doc_type'] == 'RECORD']
df_s = df_s[df_s[var] == 1]
df = None

#count co-occurring positive flags across the other category models
labels = [col for col in df_g.columns if 'pred' in col]
gen_count = [df_g[col].sum() for col in labels]
spe_count = [df_s[col].sum() for col in labels]

gen_dic_s = dict(sorted(zip(labels, gen_count), key=lambda item: item[1], reverse=True))
spe_dic_s = dict(sorted(zip(labels, spe_count), key=lambda item: item[1], reverse=True))

gen_fig_count = [gen_dic_s[k] for k in gen_dic_s]
spe_fig_count = [spe_dic_s.get(k) for k in gen_dic_s]

# Plot Histogram on x
X = list(gen_dic_s.keys())
_X = np.arange(len(X))
plt.bar(_X - 0.2, gen_fig_count, 0.4, color='b', label='General')
plt.bar(_X + 0.2, spe_fig_count, 0.4, color='g', label='Specific')
plt.xticks(rotation=90)
plt.xticks(_X, X)
plt.yscale('log')
plt.title('Prediction Distribution: {}'.format(var))
plt.legend(loc='upper right', title='Model Class:')
plt.ylabel('Count')
plt.show()